In [9]:
import xarray as xr
import matplotlib.pyplot as plt
import cartopy
import cartopy.crs as ccrs
import numpy as np
from regional_mom6 import regridding as rgd
from pathlib import Path
from matplotlib.animation import FuncAnimation
import os

In [10]:
def plot_curtain(data, var_name, key, output_dir):
    plt.figure(figsize=(10, 6))
    data.plot()
    plt.title(f'Curtain Plot of {var_name}')
    plt.savefig(os.path.join(output_dir, f'{var_name}_{key}.png'))
    plt.close()

In [11]:
# Define Segment Dir to Num
def segment_name(segment_number):
    return "segment_" + segment_number
def variable_name(variable, segment_number):
    return variable+"_"+segment_name(segment_number)
segment_num_to_dir = {
    "001":"south",
    "002":"north",
    "003":"east"
}
segment_dir_to_num =  {v: k for k, v in segment_num_to_dir.items()}

# Define Boundary Indexing Map
boundary_indexing_map = {}
for key in segment_dir_to_num.keys():
    if key == "north" or key == "south":
        boundary_indexing_map[key] = lambda arr: arr[0, :]
    elif key == "east" or key == "west":
        boundary_indexing_map[key] = lambda arr: arr[:, 0]
def get_coords_dict(hgrid):
    coords = {}
    for key in segment_dir_to_num.keys():
        coords[key] =  rgd.coords(hgrid, key, segment_name(key))
    return coords
def get_points_dict(hgrid: xr.Dataset):
    dict = {"u":None, "v":None, "t":None}
    for key in dict.keys():
        dict[key] = rgd.get_hgrid_arakawa_c_points(hgrid, key)
    return dict
def get_boundary_file_dict(file_dir, file_beginning, file_ending):
    bv = {}
    for key in segment_dir_to_num.keys():
        bv[key] = xr.open_dataset(Path(file_dir)/(file_beginning+segment_dir_to_num[key]+file_ending))
    return bv

# Hgrids
hgrid = xr.open_dataset("/glade/u/home/manishrv/documents/croc/mom_input/gfdl_copy.4.n3b/hgrid.nc")



# Read files & organize dicts
rm6_coords = get_coords_dict(hgrid)
rm6_points = get_points_dict(hgrid)


In [4]:
# OBCs
# Input File Path,must be in input MOM6 Structure
rm6_boundary_conditions = ("/glade/u/home/manishrv/scratch/croc_input/gustavo-glorys-1/ocnice")
rm6_boundary_velocities = get_boundary_file_dict(rm6_boundary_conditions,"forcing_obc_segment_",".nc" )


In [5]:
rm6_boundary_velocities["north"]

<xarray.Dataset>
Dimensions:              (time: 7670, nz_segment_002_u: 50, ny_segment_002: 1,
                          nx_segment_002: 1481, nz_segment_002_v: 50,
                          nz_segment_002_salt: 50, nz_segment_002_temp: 50,
                          nz_salt_segment_002: 50, nz_temp_segment_002: 50,
                          nz_u_segment_002: 50, nz_v_segment_002: 50, depth: 50)
Coordinates:
  * depth                (depth) float32 0.494 1.541 ... 5.275e+03 5.728e+03
  * time                 (time) object 2000-01-01 00:00:00 ... 2020-12-30 00:...
    lon_segment_002      (nx_segment_002) float64 ...
    lat_segment_002      (nx_segment_002) float64 ...
  * nz_segment_002_salt  (nz_segment_002_salt) int32 0 1 2 3 4 ... 46 47 48 49
  * nz_segment_002_temp  (nz_segment_002_temp) int32 0 1 2 3 4 ... 46 47 48 49
  * nz_segment_002_u     (nz_segment_002_u) int32 0 1 2 3 4 5 ... 45 46 47 48 49
  * nz_segment_002_v     (nz_segment_002_v) int32 0 1 2 3 4 5 ... 45 46 47 48 49
  * nx_segment_002       (nx_segment_002) int32 0 1 2 3 ... 1477 1478 1479 1480
  * ny_segment_002       (ny_segment_002) int32 0
    nyp                  int64 ...
    nxp                  (nx_segment_002) int64 ...
Dimensions without coordinates: nz_salt_segment_002, nz_temp_segment_002,
                                nz_u_segment_002, nz_v_segment_002
Data variables:
    u_segment_002        (time, nz_segment_002_u, ny_segment_002, nx_segment_002) float64 ...
    v_segment_002        (time, nz_segment_002_v, ny_segment_002, nx_segment_002) float64 ...
    eta_segment_002      (time, ny_segment_002, nx_segment_002) float32 ...
    salt_segment_002     (time, nz_segment_002_salt, ny_segment_002, nx_segment_002) float32 ...
    temp_segment_002     (time, nz_segment_002_temp, ny_segment_002, nx_segment_002) float32 ...
    dz_salt_segment_002  (time, nz_salt_segment_002, ny_segment_002, nx_segment_002) float32 ...
    dz_temp_segment_002  (time, nz_temp_segment_002, ny_segment_002, nx_segment_002) float32 ...
    dz_u_segment_002     (time, nz_u_segment_002, ny_segment_002, nx_segment_002) float32 ...
    dz_v_segment_002     (time, nz_v_segment_002, ny_segment_002, nx_segment_002) float32 ...

In [8]:
output_dir = "/glade/u/home/manishrv/documents/croc/dev/gustavo_glorys/plots/initial_obc"
for key in rm6_boundary_velocities.keys():
    for var in rm6_boundary_velocities[key].data_vars:
        if not var.startswith("dz"):
                plot_curtain(rm6_boundary_velocities[key][var][0],var,key,output_dir)


In [12]:
bc_path_2 = "/glade/derecho/scratch/manishrv/inputs_rm6/nwa_twenty_year/fix/rm6_output/2000-01-01_to_2000-01-05"
rm6_b_2 = get_boundary_file_dict(bc_path_2,"forcing_obc_segment_",".nc" )
output_dir = "/glade/u/home/manishrv/documents/croc/dev/gustavo_glorys/plots/second_try_obc_expand_to_1"

for key in rm6_b_2.keys():
    for var in rm6_b_2[key].data_vars:
        if not var.startswith("dz"):
                plot_curtain(rm6_b_2[key][var][0],var,key,output_dir)

In [14]:
east_ds = xr.open_dataset("/glade/derecho/scratch/manishrv/inputs_rm6/nwa_twenty_year/fix/rm6_output/2000-01-01_to_2000-01-05/forcing_obc_segment_003.nc")

In [18]:
np.where(np.isnan(east_ds.salt_segment_003.values))

(array([], dtype=int64),
 array([], dtype=int64),
 array([], dtype=int64),
 array([], dtype=int64))